# 05 - Real Open-Source RAG + LLM Dataset (Bitext Customer Support)

Builds this repo's **RAG knowledge base** and **LLM SFT dataset** from the
open-source
[`bitext/Bitext-customer-support-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)
(instruction + category + intent + agent response), replacing
`data/sample/knowledge_base.json` / `data/sample/sft_dataset.json` with
real-data equivalents.

**Designed for a free Colab/Kaggle GPU runtime** for the fine-tuning cell.
Set `MAX_SAMPLES` to a small number (e.g. `300`) to smoke-test everything
on a laptop/CPU first.


## 1. Setup (Colab only -- skip if running locally)

In [ ]:
# Uncomment on a fresh Colab runtime:
# !git clone https://github.com/<your-username>/AI-support-assistant.git
# %cd AI-support-assistant
# !pip install -q -r requirements.txt


## 2. Configuration

- `MAX_SAMPLES`: set to e.g. `300`-`1000` for a low-compute smoke test on CPU;
  set to `None` to build the knowledge base / SFT dataset from the full
  dataset (~26.8k rows).


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

MAX_SAMPLES = 300  # set to None for the full dataset
OUTPUT_DIR = Path("data/processed")


## 3. Pull the dataset and build the knowledge base + SFT dataset

`src.data_engineering.support_dataset_builder.build` pulls the Bitext
dataset, deduplicates one canonical document per intent into a RAG
knowledge base, and builds (instruction, context, query, response) SFT
pairs grounded in that knowledge base -- in the same schema as
`data/sample/knowledge_base.json` / `data/sample/sft_dataset.json`.


In [ ]:
from src.data_engineering.support_dataset_builder import build

build(max_samples=MAX_SAMPLES, output_dir=OUTPUT_DIR)


## 4. Sanity check: retrieval over the real knowledge base

In [ ]:
from src.data_engineering.ingestion import load_documents_from_directory
from src.rag.hybrid_search import HybridRetriever

documents = load_documents_from_directory(OUTPUT_DIR)
print(f"{len(documents)} knowledge-base documents")

retriever = HybridRetriever(
    doc_ids=[d.doc_id for d in documents],
    texts=[d.text for d in documents],
    metadatas=[d.metadata for d in documents],
)
results = retriever.search("I want to cancel my order", top_k=3)
[(r.doc_id, round(r.score, 3)) for r in results]


## 5. Fine-tune the LLM (QLoRA) on the real SFT dataset

Points `src.llm.finetuning.train` at `bitext_sft_dataset.json` instead of
the bundled sample. Requires a GPU (free Colab/Kaggle T4 works for a small
base model).


In [ ]:
from src.llm.finetuning import train

train(
    base_model="Qwen/Qwen2.5-3B-Instruct",
    data_path=str(OUTPUT_DIR / "bitext_sft_dataset.json"),
    output_dir="checkpoints/qlora-bitext-support-assistant",
    epochs=1 if MAX_SAMPLES else 3,
)


## 6. Evaluate

Reuse `src.llm.evaluation` the same way `04_llm_finetuning_comparison.ipynb`
does, pointing at the real held-out split instead of the sample dataset.
